In [1]:
import pandas as pd
import numpy as np
import requests
import time
import mygene
import myvariant

In [2]:
df = pd.read_csv("datosGene4PD/t_common_variant.txt", sep = "\t", index_col = False)

In [3]:
df

,Chr,gene_symbol,SNPs_symbol,SNP_position,effect_allele,alternate_allele,joint_phase_P,joint_phase_OR,joint_phase_OR_CI,pubMed_ID,Unnamed: 10
0,6,GPR126,rs757765789,142758601,T,G,1.42E-06,1.06,1.016-1.098,28256260,NaN
1,1,SYT11,rs202015799,155839054,C,T,4.70E-09,-,-,24842889,NaN
2,12,SLC2A13,rs1994090,40428561,G,T,3.20E-54,12.05,8.35-17.41,24842889,NaN
3,12,SLC2A13,rs2708453,40478652,G,T,3.62E-54,12.05,8.35-17.41,24842889,NaN
4,12,SLC2A13,rs4768212,40474147,C,T,3.62E-54,12.05,8.35-17.41,24842889,NaN
...,...,...,...,...,...,...,...,...,...,...,...
1053,rs117896735,INPP5F,13,U,13,U,1.21E-11,1.77,-,29700661,NaN
1054,rs12456492,RIT2,21,U,21,U,2.15E-11,1.1,-,29700661,NaN
1055,rs7155501,GCH1,15,U,15,U,1.25E-10,1.12,-,29700661,NaN
1056,rs10797576,SIPA1L2,21,U,21,U,1.76E-10,1.13,-,29700661,NaN


In [4]:
df_filtrado = df[df["SNPs_symbol"].str.startswith('rs')].reset_index(drop = True)

In [5]:
df_filtrado = df_filtrado.drop(["Unnamed: 10"], axis = 1)

In [6]:
df_filtrado = df_filtrado.dropna(subset = ['gene_symbol']).reset_index(drop = True)

In [21]:
mv = myvariant.MyVariantInfo()

In [22]:
prueba = mv.querymany(['rs757765789'], scopes = "dbsnp.rsid", fields = "dbsnp", species = "human")

In [27]:
prueba[0]["dbsnp"]["gene"]["strand"]

'+'

In [19]:
def busca_rsIDs(df):
    
    lista_rsids = df["SNPs_symbol"].unique().tolist()

    mv = myvariant.MyVariantInfo()

    resultados = mv.querymany(lista_rsids, scopes = "dbsnp.rsid", fields = "dbsnp", species = "human")

    diccionario_snps = {}

    cromosomas_validos = [str(i) for i in range(1, 23)] + ["X", "Y", "MT"]

    bases_validas = ["A", "C", "G", "T", "a", "c", "g", "t"]

    cadenas_validas = ["+", "-", "1", "-1"]

    for resultado in resultados:
        rsid = resultado.get("query")

        if "notfound" in resultado:
            continue

        info_dbsnp = resultado.get("dbsnp", {})

        hg19 = info_dbsnp.get("hg19", {})

        if not hg19:
            continue

        if isinstance(hg19, list):
            hg19 = hg19[0]

        cromosoma = str(info_dbsnp.get("chrom"))
        if cromosoma not in cromosomas_validos:
            continue

        pos_hg19 = hg19.get("start")
        
        cadena = str(info_dbsnp.get("strand"))

        if not cadena:
            continue

        ef_allele = info_dbsnp.get("ref", '')
        alt_allele = info_dbsnp.get("alt", '')

        if ef_allele not in bases_validas or alt_allele not in bases_validas:
            continue

        gene_info = info_dbsnp.get("gene", {})
        gene_symbol = "inter"

        if isinstance(gene_info, dict):
            gene_symbol = gene_info.get("symbol", "inter")

        elif isinstance(gene_info, list):
            gene_symbol = gene_info[0].get("symbol", "inter")

        if rsid not in diccionario_snps:

            diccionario_snps[rsid] = {"SNPs_symbol": rsid, "Chr_corr": cromosoma, "SNP_position_corr": pos_hg19, "Effect_allele_corr": ef_allele, "Alternate_allele_corr": [alt_allele], "Gene_symbol_corr": gene_symbol, "Cadena": cadena}

        else:

            if alt_allele not in diccionario_snps[rsid]["Alternate_allele_corr"]:
                diccionario_snps[rsid]["Alternate_allele_corr"].append(alt_allele)

    datos_corregidos = list(diccionario_snps.values())

    return pd.DataFrame(datos_corregidos)

In [20]:
df_corr = busca_rsIDs(df_filtrado)

ConnectError: [Errno 11001] getaddrinfo failed

In [9]:
df_corr

,SNPs_symbol,Chr_corr,SNP_position_corr,Effect_allele_corr,Alternate_allele_corr,Gene_symbol_corr
0,rs757765789,6,142758601,T,[G],ADGRG6
1,rs202015799,1,155839054,C,"[G, T]",SYT11
2,rs1994090,12,40428561,G,"[A, T, C]",SLC2A13
3,rs2708453,12,40478652,G,"[A, T]",SLC2A13
4,rs4768212,12,40474147,C,"[A, T]",SLC2A13
...,...,...,...,...,...,...
881,rs666463,17,76425480,A,[T],DNAH17
882,rs1941685,18,31304318,G,"[T, C]",ASXL3
883,rs8087969,18,48683589,T,"[G, A]",inter
884,rs77351827,20,6006041,C,[T],CRLS1
